# [Unified] NB01/04 — Teacher 학습데이터 생성 (agent/forward 공용, v3: EXPERIMENT_STAGE)

**EXPERIMENT_STAGE 4단계** (Bootstrap 셀 한 줄로 전환):

| Stage | task | prompt | rejection | agent keys | 목적 (실험 서사) |
|---|---|---|---|---|---|
| S1 | forward | v1 | OFF | — | rejection 없는 SFT: 과제 설계 문제 재현 (고유율 18.9% / conf std<10) |
| S2 | forward | v2 | ON | — | rejection + v2: forward 성공 (고유율 75.8% / ECE 0.109) — 배포 후보 |
| S3 | agent | v2 | ON | 3key | 3-key evaluator: 데이터 완벽 / student 붕괴 재현 (coupling 0위반) |
| S4 | agent | v2 | ON | 1key | 1-key wm + det cls/suf: 현행 (GRPO는 NB02) |

**`TEACHER_TASK` 한 줄로 전환** (F1-b 셀). 샘플링·z-table·balance·eval분리·진단은 완전 동일,
prompt(prompts.py)·key·rejection 기준만 task로 분기.

- forward → `{cls, confidence, reasoning}`, `training_data_forward*.jsonl`
- agent   → `{cls, what_missing}`, `training_data_mined*.jsonl`


## Section 0: 환경 (GCP 인증 / gcsfuse / 패키지)

In [1]:
# GCP 인증 — gcsfuse 마운트를 위해 여전히 필요
# (Groq API는 별도 API key로 동작, GCP 인증과 무관)
from google.colab import auth
auth.authenticate_user()
print('✅ GCP 인증 완료 (gcsfuse 마운트용)')

✅ GCP 인증 완료 (gcsfuse 마운트용)


In [2]:
# 2. gcsfuse 설치
!echo "deb http://packages.cloud.google.com/apt gcsfuse-`lsb_release -c -s` main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!sudo apt-get update
!sudo apt-get install gcsfuse

deb http://packages.cloud.google.com/apt gcsfuse-jammy main
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1022  100  1022    0     0  12770      0 --:--:-- --:--:-- --:--:-- 12936
OK
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://packages.cloud.google.com/apt gcsfuse-jammy InRelease [1,227 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,845 kB]
Get:8 http://packages.cloud.google.com/apt gcsfuse-jammy/main amd64 Packages [73.2 kB]
Hit:9 http://archive.ubunt

In [3]:
# gcsfuse 설치 & 버킷 마운트
import os

GCS_BUCKET  = "fraud_ecomm"
GCP_PROJECT = "kakaoo-501107"
MOUNT_POINT = "/content/gcs_mount"

!apt-get install -y -q gcsfuse

os.makedirs(MOUNT_POINT, exist_ok=True)

# gcsfuse 마운트 (ADC 자격증명 자동 사용)
!gcsfuse --implicit-dirs \
    --key-file="" \
    {GCS_BUCKET} {MOUNT_POINT} 2>&1 | tail -3

# 마운트 확인
result = os.listdir(MOUNT_POINT)
print(f'✅ GCS 마운트 완료: {MOUNT_POINT}')
print(f'   최상위 폴더: {result[:5]}')

Reading package lists...
Building dependency tree...
Reading state information...
gcsfuse is already the newest version (3.11.1).
0 upgraded, 0 newly installed, 0 to remove and 148 not upgraded.
{"timestamp":{"seconds":1784715051,"nanos":371915237},"severity":"INFO","message":"GCSFuse Config","mount-id":"fraud_ecomm-7a24cb05","CLI Flags":{"implicit-dirs":"true","key-file":""}}
{"timestamp":{"seconds":1784715051,"nanos":371961491},"severity":"INFO","message":"GCSFuse Config","mount-id":"fraud_ecomm-7a24cb05","Full Config":{"AppName":"","CacheDir":"","CloudProfiler":{"AllocatedHeap":true,"Cpu":true,"Enabled":false,"Goroutines":false,"Heap":true,"Label":"gcsfuse-0.0.0","Mutex":false,"ServiceName":"gcsfuse"},"Debug":{"ExitOnInvariantViolation":false,"Fuse":false,"Gcs":false,"LogMutex":false},"DisableAutoconfig":false,"DummyIo":{"Enable":false,"PerMbLatency":0,"ReaderLatency":0},"EnableAtomicRenameObject":true,"EnableGoogleLibAuth":true,"EnableHns":true,"EnableNewReader":true,"EnableStandar

In [4]:
# v5: 공통 패키지 일괄 설치 (provider 무관)
!pip install -q \
    trl peft transformers accelerate datasets \
    groq google-genai \
    bitsandbytes

# !pip install flash-attn --no-build-isolation -q #flash attention 일단 제거
# google-generativeai
print("✅ 설치 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 49.5 MB/s eta 0:00:00
✅ 설치 완료


In [5]:
# os.environ["HF_HUB_DISABLE_XET"] = "1"
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="1"

# !pip -q install hf_transfer

In [6]:
# !pip install huggingface_hub==1.23.0

In [7]:
# import huggingface_hub; print(huggingface_hub.__version__)

## Bootstrap: 공통 모듈 + 데이터 로드

In [8]:
# ============================================================
# 공통 모듈(common/*.py) 로드 — 버킷에서 로컬로 복사 후 import
# (편집-반영 안정 + import 캐시 회피). 버킷 경로는 환경에 맞게 1회 수정.
# ============================================================
import sys, os, shutil, importlib

COMMON_SRC = "/content/gcs_mount/tree-search/analyzer_ft/common"   # ← .py 업로드 위치
COMMON_DST = "/content/common"
shutil.rmtree(COMMON_DST, ignore_errors=True)
shutil.copytree(COMMON_SRC, COMMON_DST)
if COMMON_DST not in sys.path:
    sys.path.insert(0, COMMON_DST)

import config, prompts, data_io, schema
for _m in (config, prompts, data_io, schema):
    importlib.reload(_m)
print("✅ common 모듈 로드:", [m.__name__ for m in (config, prompts, data_io, schema)])


✅ common 모듈 로드: ['config', 'prompts', 'data_io', 'schema']


In [9]:
# ============================================================
# 데이터 로드 + 런타임 바인딩 (mode="agent")
# config 상수를 펼치고, data_io.load_all()이 데이터/맵을 주입.
# ============================================================
from config import *
PHASE = "generate"
GENERATION_MODE = "mining_only"
config.ensure_dirs()
config.print_config()

# ════════ EXPERIMENT_STAGE — 4단계 실험 스위치 (전반적인 문제 해결 단계) ════════
#  S1: forward + prompt v1 + rejection OFF + teacher confidence 원본 (문제 발견 재현)
#  S2: forward + prompt v2 + rejection ON  + nd-relabel confidence (배포 후보)
#  S3: agent 3-key {cls,sufficiency,what_missing} — ambiguous cls는 teacher evidence(GT-rejection)
#  S4: agent 1-key {what_missing} — cls/sufficiency 전부 nd 결정론 (현행)
EXPERIMENT_STAGE = "S1"     # ★ "S1" | "S2" | "S3" | "S4"
_STAGE_CFG = {
    "S1": dict(task="forward", prompt_ver="v1", rejection=False, conf_relabel=False,
               agent_keys=None,   stem="training_data_forward_s1"),
    "S2": dict(task="forward", prompt_ver="v2", rejection=True,  conf_relabel=True,
               agent_keys=None,   stem="training_data_forward"),
    "S3": dict(task="agent",   prompt_ver="v2", rejection=True,  conf_relabel=None,
               agent_keys="3key", stem="training_data_mined_3key"),
    "S4": dict(task="agent",   prompt_ver="v2", rejection=True,  conf_relabel=None,
               agent_keys="1key", stem="training_data_mined"),
}[EXPERIMENT_STAGE]
TEACHER_TASK     = _STAGE_CFG["task"]
STAGE_PROMPT_VER = _STAGE_CFG["prompt_ver"]
REJECTION_ON     = _STAGE_CFG["rejection"]      # False(S1) → cls-GT/grounded/dup 필터 해제
CONF_RELABEL     = _STAGE_CFG["conf_relabel"]   # False(S1) → teacher confidence 원본 유지
AGENT_KEYS       = _STAGE_CFG["agent_keys"]     # "3key"(S3) | "1key"(S4)
STAGE_STEM       = _STAGE_CFG["stem"]           # 산출물 stem — stage별 파일 분리 (stale 방지)
print(f"■ EXPERIMENT_STAGE={EXPERIMENT_STAGE}  {_STAGE_CFG}")

globals().update(data_io.load_all(mode="agent"))   # test_df, labels, coreset_* 등 펼침
SAVE_TEACHER_TO_GCS = True   # ★ 저장 스위치

from prompts import (SYSTEM_PROMPT, build_sample, extract_test_features,
                     build_analyzer_user_prompt, build_teacher_prompt_with_gt, build_teacher_prompt_gtaware_agent,
                     compute_l2_distances, format_top5_neighbors,
                     format_transaction, format_persona, normalize_dist, dist_tag)

_is_think = any(t in HF_TEACHER_MODEL for t in ("QwQ", "Thinking"))
print("✅ 런타임 준비 완료 (build_sample / build_analyzer_user_prompt 사용 가능)")


PHASE            : generate
EXPERIMENT_CASE  : pca_64_k100
TEACHER_PROVIDER : huggingface
TEACHER_MODEL    : Qwen/Qwen2.5-14B-Instruct
GENERATION_MODE  : mining_only
TRAIN_ON         : mining_only
TRAINING_JSONL   : 
OUT_ADAPTER_PATH : /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/adapter
■ EXPERIMENT_STAGE=S1  {'task': 'forward', 'prompt_ver': 'v1', 'rejection': False, 'conf_relabel': False, 'agent_keys': None, 'stem': 'training_data_forward_s1'}
✅ data_io.load_all(mode=agent)
   personas=100  coreset_map=122,810
   test=28,302  normal/abnormal=14151/14151
   dist p25/p50/p75=1.264/1.448/1.651  IQR=0.387
✅ 런타임 준비 완료 (build_sample / build_analyzer_user_prompt 사용 가능)


## (NB01 재사용) boundary 샘플러 + Teacher 초기화

In [10]:
def sample_boundary_distance(low=0.3, high=0.5, target=100,
                              exclude_first_pct=0.0, exclude_last_pct=0.0):
    """경계 distance 구간(0.3~0.5)의 샘플만 추출 — LLM이 담당할 모호 영역.
       NORMAL/ABNORMAL 둘 다 포함(이 구간이 진짜 어려운 곳)."""
    n_total = len(test_df)
    a = int(n_total * exclude_first_pct)
    b = n_total - int(n_total * exclude_last_pct)
    cand = []
    for i in range(a, b):
        nd = normalize_dist(compute_l2_distances(test_emb[i],
                            [int(top5_indices[i][0])])[0])
        if low <= nd < high:
            cand.append((i, "boundary_distance"))
    import random as _r
    _r.Random(42).shuffle(cand)
    sel = cand[:target]
    n_ab = sum(1 for i,_ in sel if labels[i]==1)
    print(f"  [boundary {low}~{high}] {len(cand)}개 중 {len(sel)}개 선택 (ABNORMAL {n_ab})")
    return sel

## Section 8: Teacher 초기화

In [11]:
# ============================================================
# Teacher 호출 함수 (provider 자동 분기)
# ============================================================
from schema import validate_agent_output
import re

# def _parse_and_validate(text: str) -> Dict:
#   # QwQ-32B 등 reasoning 모델의 <think> 태그 제거
#     if "<think>" in text:
#         text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
#     if "```json" in text: text = text.split("```json")[-1].split("```")[0].strip()
#     elif "```" in text:   text = text.split("```")[1].strip()
#     parsed = validate_agent_output(json.loads(text))      # required: cls, sufficiency
#     parsed["cls"] = str(parsed["cls"]).upper()
#     if parsed["cls"] not in ("NORMAL", "ABNORMAL"):
#         raise ValueError(f"Invalid cls: {parsed['cls']}")
#     na = parsed.get("next_action") or "none"
#     parsed["next_action"] = na if na in ("none", "expand", "reroute") else "none"
#     return parsed
def _parse_and_validate(text: str) -> Dict:
    if "</think>" in text:   text = text.split("</think>")[-1]
    elif "<think>" in text:  text = text.split("<think>")[0]
    text = text.strip()
    if "```json" in text: text = text.split("```json")[-1].split("```")[0].strip()
    elif "```" in text:   text = text.split("```")[1].strip()
    if "{" in text and "}" in text:
        text = text[text.index("{"): text.rindex("}")+1]
    else:
        raise ValueError("no JSON object (truncated think)")   # ← 추가
    if not text.strip():
        raise ValueError("empty (truncated think)")
    parsed = validate_agent_output(json.loads(text))
    parsed["cls"] = str(parsed["cls"]).upper()
    if parsed["cls"] not in ("NORMAL","ABNORMAL"):
        raise ValueError(f"Invalid cls: {parsed['cls']}")
    na = parsed.get("next_action") or "none"
    parsed["next_action"] = na if na in ("none","expand","reroute") else "none"
    return parsed
# ── Groq ──────────────────────────────────────────────────
def _call_groq(user_prompt: str, model: str, max_retry: int = 5) -> Optional[Dict]:
    for attempt in range(max_retry):
        try:
            resp = groq_client.chat.completions.create(
                model=model,
                messages=[{"role":"system","content":SYSTEM_PROMPT},
                          {"role":"user","content":user_prompt}],
                temperature=0.7, max_tokens=max_new_tokens,
                response_format={"type":"json_object"},
            )
            return _parse_and_validate(resp.choices[0].message.content)
        except Exception as e:
            err = str(e)
            if ("tokens per day" in err or "TPD" in err) and "429" in err:
                fallbacks = [m for m in GROQ_FALLBACK_MODELS if m != model]
                if fallbacks:
                    print(f"  → TPD 소진 ({model}), 즉시 fallback: {fallbacks[0]}")
                    return _call_groq(user_prompt, fallbacks[0], max_retry)
                return None
            wait = 3 if ("429" in err or "rate_limit" in err.lower()) else 2
            print(f"  [groq retry {attempt+1}/{max_retry}] {err[:60]} — {wait}s")
            time.sleep(wait)
    return None

# ── Gemini ────────────────────────────────────────────────
def _call_gemini(user_prompt: str, model_name: str, max_retry: int = 5) -> Optional[Dict]:
    import google.generativeai as genai
    for attempt in range(max_retry):
        try:
            mdl = genai.GenerativeModel(
                model_name=model_name,
                system_instruction=SYSTEM_PROMPT,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.7, max_output_tokens=1024,
                    response_mime_type="application/json"),
            )
            return _parse_and_validate(mdl.generate_content(user_prompt).text)
        except Exception as e:
            err = str(e)
            is_quota = ("429" in err or "RESOURCE_EXHAUSTED" in err
                        or "quota" in err.lower() or "rate" in err.lower())
            if is_quota and attempt >= 1:
                fallbacks = [m for m in GEMINI_FALLBACK_MODELS if m != model_name]
                if fallbacks:
                    print(f"  → Rate limit ({model_name}), 즉시 fallback: {fallbacks[0]}")
                    return _call_gemini(user_prompt, fallbacks[0], max_retry)
                return None
            wait = 3 if is_quota else 2
            print(f"  [gemini retry {attempt+1}/{max_retry}] {err[:60]} — {wait}s")
            time.sleep(wait)
    return None

# ── HuggingFace 배치 추론 ─────────────────────────────────
def _hf_batch_infer(prompts: List[str], batch_size: int = HF_BATCH_SIZE) -> List[Optional[Dict]]:
    from tqdm.auto import tqdm
    import torch
    global hf_model, hf_tokenizer, _hf_current_model

    results: List[Optional[Dict]] = [None] * len(prompts)
    # _brief = ("\n\nThink briefly (2-3 sentences max), then output the JSON."
    #           if ("QwQ" in _hf_current_model or "Thinking" in _hf_current_model) else "")
    _brief = ("\n\nThink briefly (2-3 sentences max), then output the JSON."
          if any(t in _hf_current_model for t in ("QwQ", "Thinking")) else "")
    chats = []
    for p in prompts:
        chats.append([
            {"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content": p + _brief + "\n\nIMPORTANT: Respond ONLY with valid JSON, no markdown."},
        ])

    n_batches = (len(chats) + batch_size - 1) // batch_size
    n_ok = n_fail = 0

    # ▼ 변경: 배치 루프에 tqdm 부착
    pbar = tqdm(range(0, len(chats), batch_size), total=n_batches,
                desc="HF 배치 추론", unit="batch")
    for batch_start in pbar:
        batch = chats[batch_start : batch_start + batch_size]
        texts = [
            hf_tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=True
            )
            for c in batch
        ]
        enc = hf_tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1536,
        ).to(hf_model.device)

        with torch.no_grad():
            out_ids = hf_model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                pad_token_id=hf_tokenizer.pad_token_id,
                eos_token_id=hf_tokenizer.eos_token_id,
            )

        input_len = enc["input_ids"].shape[1]
        for j, out in enumerate(out_ids):
            raw = hf_tokenizer.decode(out[input_len:], skip_special_tokens=True)
            try:
                results[batch_start + j] = _parse_and_validate(raw)
                n_ok += 1
            except Exception as e:
                print(f"  ⚠ parse error [{batch_start+j}]: {str(e)[:60]}")
                results[batch_start + j] = None
                n_fail += 1
        # ▼ 배치 완료 후 ok/fail 현황 갱신
        pbar.set_postfix(ok=n_ok, fail=n_fail,
                         done=f"{batch_start+len(batch)}/{len(prompts)}")

    return results

# ── 단건 진입점 (Groq/Gemini용) ──────────────────────────
def call_teacher(user_prompt: str) -> Optional[Dict]:
    if TEACHER_PROVIDER == "groq":
        return _call_groq(user_prompt, GROQ_TEACHER_MODEL)
    elif TEACHER_PROVIDER == "gemini":
        return _call_gemini(user_prompt, GEMINI_TEACHER_MODEL)
    else:  # huggingface: 배치용 함수 사용 권장, 여기서는 단건 래퍼
        results = _hf_batch_infer([user_prompt], batch_size=1)
        return results[0]

# ── 학습 데이터 생성 (HF는 배치, API는 sequential) ────────
def generate_training_data(sampled: List[Tuple[int, str]]) -> List[Dict]:
    # from tqdm.auto import tqdm

    n_total = len(sampled)
    est = (f"~{n_total/TEACHER_RPM:.0f}분 예상" if TEACHER_RPM
           else f"batch={HF_BATCH_SIZE}, 속도제한 없음")
    print(f"학습 데이터 생성 시작 (총 {n_total}개 | {est})")

    # # ── 공통: 입력 준비 ─────────────────────────────────
    teacher_prompts, analyzer_prompts, metas = [], [], []
    for n, (test_idx, category) in enumerate(sampled):
        gt_label   = int(labels[test_idx])
        cluster_id = int(gt_clusters[test_idx])
        top5_oi    = [int(x) for x in top5_indices[test_idx]]
        persona    = CLUSTER_PERSONAS.get(cluster_id)
        tf         = extract_test_features(test_idx)
        dists      = compute_l2_distances(test_emb[test_idx], top5_oi)

        # ✅ top1/top3 50:50 (인덱스 짝/홀로 결정 → 재현성)
        topk = EVIDENCE_TOPK_CHOICES[n % len(EVIDENCE_TOPK_CHOICES)]

        # # ✅ teacher엔 GT hint 주입(1-A 수정), user(analyzer)엔 미포함
        # teacher_prompts.append(
        #     build_teacher_prompt_with_gt(tf, cluster_id, top5_oi, dists, gt_label,
        #                                  persona, include_gt_hint=True, top_k=topk)
        # )

        # ✅ Rejection sampling: GT hint 없이 순수 추론 → 아래 필터가 GT로 거름
        teacher_prompts.append(
            build_teacher_prompt_with_gt(tf, cluster_id, top5_oi, dists, gt_label,
                                         persona, include_gt_hint=False, top_k=topk)
        )

        analyzer_prompts.append(
            build_analyzer_user_prompt(tf, cluster_id, top5_oi, dists, persona, top_k=topk)
        )
        metas.append({
            "test_idx":test_idx, "gt_label":gt_label, "category":category,
            "cluster_id":cluster_id, "top_k":topk,                      # ✅ 기록
            "user_prompt":analyzer_prompts[-1],                         # ✅ GRPO용(3-A 수정)
            "top1_l2_distance":float(dists[0]) if dists else -1.0,
            "top1_l2_dist_norm":normalize_dist(dists[0]),
            "topk_spread": float(np.std(dists[:3])) if len(dists) > 1 else 0.0,
            "dist_p50":DIST_P50, "dist_iqr":DIST_IQR,
            "experiment_case":EXPERIMENT_CASE,
            "teacher_model":TEACHER_MODEL, "teacher_provider":TEACHER_PROVIDER,
        })
    # ── HF: 배치 추론 ────────────────────────────────────
    if TEACHER_PROVIDER == "huggingface":
        t0 = time.time()
        outputs = _hf_batch_infer(teacher_prompts, batch_size=HF_BATCH_SIZE)
        elapsed = time.time() - t0

        records = []
        n_ok = n_fail = n_gtmiss = 0
        for i, output in enumerate(outputs):
            if output is None:
                n_fail += 1
                continue
            # ✅ GT 불일치 필터 (teacher가 GT 틀린 샘플은 학습서 제외)
            pred_lab = 1 if str(output.get("cls","")).upper() == "ABNORMAL" else 0
            if pred_lab != metas[i]["gt_label"]:
                n_gtmiss += 1
                continue
            records.append({"system":SYSTEM_PROMPT, "user":analyzer_prompts[i],
                            "assistant":json.dumps(output, ensure_ascii=False),
                            "metadata":metas[i]})
            n_ok += 1
        # === Pass 2: 채택분에 GT-aware what_missing 재생성 (루프 밖, 1회만) ===
        if records:
            wm_prompts = []
            for r in records:
                ti = r["metadata"]["test_idx"]
                oi = [int(x) for x in top5_indices[ti]]
                wm_prompts.append(build_teacher_prompt_gtaware_agent(
                    extract_test_features(ti), r["metadata"]["cluster_id"], oi,
                    compute_l2_distances(test_emb[ti], oi),
                    r["metadata"]["gt_label"],
                    CLUSTER_PERSONAS.get(r["metadata"]["cluster_id"]),
                    top_k=r["metadata"]["top_k"]))
            wm_out = _hf_batch_infer(wm_prompts, batch_size=HF_BATCH_SIZE)
            n_wm = 0
            for r, o in zip(records, wm_out):
                if o and o.get("what_missing"):
                    a = json.loads(r["assistant"])
                    a["what_missing"] = o["what_missing"]
                    a["reason"]       = o.get("reason", a.get("reason", ""))
                    r["assistant"]    = json.dumps(a, ensure_ascii=False)
                    n_wm += 1
            print(f"  [Pass2] GT-aware what_missing 채움: {n_wm}/{len(records)}")

        print(f"\n✅ 생성 완료: {n_ok}/{n_total} 성공 (parse실패 {n_fail}, GT불일치 {n_gtmiss})  총 {elapsed:.0f}s")
        return records

    # ── API (Groq/Gemini): Sequential + rate limit ───────
    interval = 60.0 / TEACHER_RPM
    records  = []
    n_ok = n_fail = n_gtmiss = 0
    t_start = time.time()
    for ap, meta, tp in zip(analyzer_prompts, metas, teacher_prompts):
        t_req = time.time()
        output = call_teacher(tp)
        if output is None:
            n_fail += 1
        else:
            pred_lab = 1 if str(output.get("cls","")).upper() == "ABNORMAL" else 0
            if pred_lab != meta["gt_label"]:
                n_gtmiss += 1
            else:
                records.append({"system":SYSTEM_PROMPT, "user":ap,
                                "assistant":json.dumps(output, ensure_ascii=False),
                                "metadata":meta})
                n_ok += 1
        time.sleep(max(0, interval - (time.time() - t_req)))
    print(f"\n✅ 생성 완료: {n_ok}/{n_total} 성공 (parse실패 {n_fail}, GT불일치 {n_gtmiss})")
    return records

print("✅ Teacher 함수 정의 완료")
print(f"   Provider: {TEACHER_PROVIDER.upper()} | HF batch_size={HF_BATCH_SIZE if TEACHER_PROVIDER=='huggingface' else 'N/A'}")

✅ Teacher 함수 정의 완료
   Provider: HUGGINGFACE | HF batch_size=50


In [12]:
# ── PHASE 가드 ────────────────────────────────────────────
if PHASE == "train":
    print("ℹ️  PHASE='train': Teacher 초기화 skip → Section 10으로 이동")
else:
    # ── 전역 변수 초기화 ──────────────────────────────────
    groq_client       = None
    gemini_client     = None
    hf_model          = None   # 모델 직접 참조 (GPU 해제용)
    hf_tokenizer      = None
    hf_pipeline       = None
    _hf_current_model = None

    # os.environ["HF_HUB_DISABLE_XET"] = "0"
    # os.environ["HF_XET_HIGH_PERFORMANCE"]="0"

    if TEACHER_PROVIDER == "groq":
        from groq import Groq
        groq_client = Groq(api_key=GROQ_API_KEY)
        print(f"✅ Groq  | model={TEACHER_MODEL}  RPM={TEACHER_RPM}")

    elif TEACHER_PROVIDER == "gemini":
        # import google.generativeai as genai
        from google import genai
        client = genai.Client(api_key=GEMINI_API_KEY)
        # genai.configure(api_key=GEMINI_API_KEY)
        print(f"✅ Gemini | model={TEACHER_MODEL}  RPM={TEACHER_RPM}")
        print(f"   Fallback: {GEMINI_FALLBACK_MODELS}")

    elif TEACHER_PROVIDER == "huggingface":
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

        def _load_hf_model(model_id: str, use_4bit: bool = HF_USE_4BIT):
            """HuggingFace 모델 + 토크나이저 로드. (model, tokenizer) 반환."""
            tok = AutoTokenizer.from_pretrained(
                model_id, token=HF_TOKEN or None,
                padding_side="left",   # 배치 추론: left-padding 필
            )
            if tok.pad_token is None:
                tok.pad_token = tok.eos_token

            model_kwargs = dict(device_map="auto", trust_remote_code=True,
                                token=HF_TOKEN or None)
            if model_id.upper().endswith("AWQ"):
              # AWQ 모델은 이미 양자화되어 있으므로 BnB 설정을 사용하지 않음
                model_kwargs["torch_dtype"] = "auto"
            elif use_4bit:
                model_kwargs["quantization_config"] = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4",
                )
                print(f"  ⚙ 4-bit NF4 양자화 적용")
            else:
                model_kwargs["torch_dtype"] = torch.bfloat16
            mdl = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
            mdl.eval()
            return mdl, tok

        print(f"[HF] 모델 로드 중: {HF_TEACHER_MODEL} (batch_size={HF_BATCH_SIZE}) ...")
        try:
            hf_model, hf_tokenizer = _load_hf_model(HF_TEACHER_MODEL)
            _hf_current_model = HF_TEACHER_MODEL
            mem_gb = torch.cuda.memory_allocated() / 1024**3
            print(f"✅ HF | model={_hf_current_model}")
            print(f"   4-bit={HF_USE_4BIT}  batch_size={HF_BATCH_SIZE}")
            print(f"   GPU 사용: {mem_gb:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
            print(f"   Rate limit: 없음 (로컬 추론)")
        except Exception as e:
            print(f"⚠ 기본 모델 로드 실패: {e}")
            if HF_FALLBACK_MODELS:
                fb = HF_FALLBACK_MODELS[0]
                print(f"  → Fallback: {fb}")
                hf_model, hf_tokenizer = _load_hf_model(fb)
                _hf_current_model = fb
                print(f"✅ Fallback 로드 완료: {_hf_current_model}")

[HF] 모델 로드 중: Qwen/Qwen2.5-14B-Instruct (batch_size=50) ...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ HF | model=Qwen/Qwen2.5-14B-Instruct
   4-bit=True  batch_size=50
   GPU 사용: 9.25 GB / 39.5 GB
   Rate limit: 없음 (로컬 추론)


## Section F1: Deviation Table + 3-key 프롬프트/파서
**생성 원리** — "무엇이 이탈했는지"는 코드가 계산(Deviation Table), teacher는 그 테이블을
근거로 서술만 한다. Deterministic-LLM 역할 분리 원칙을 **데이터 생성 단계에 적용**:
boilerplate("close to cluster range" 류)가 구조적으로 나올 이유가 사라지고,
grounded-rejection(F3)의 검증 대상이 명시적으로 존재하게 된다.

In [13]:
# ── F1-a. Deviation Table: persona 대비 feature별 이탈 신호 (deterministic) ──
Z_TAU = 1.0   # numeric 이탈 판정 |z| 기준 (grounded-rejection에서도 동일 사용)

def _persona_numeric(persona):  return (persona or {}).get("numeric_stats", {}) or {}
def _persona_cat(persona):      return (persona or {}).get("categorical_distribution", {}) or {}

def build_deviation_table(test_features: dict, persona) -> dict:
    """
    반환: {"numeric": {col: {"value","z"}}, "categorical": {col: {"value","in_profile"}},
           "deviant": [이탈 feature명...], "conforming": [...]}
    """
    out = {"numeric": {}, "categorical": {}, "deviant": [], "conforming": []}
    for col, st in _persona_numeric(persona).items():
        if col not in test_features: continue
        try:
            v = float(test_features[col]); mu = float(st.get("mean", 0.0))
            sd = float(st.get("std", 0.0)) or 1e-6
            z = (v - mu) / sd
        except (TypeError, ValueError):
            continue
        out["numeric"][col] = {"value": round(v, 3), "z": round(z, 2)}
        (out["deviant"] if abs(z) >= Z_TAU else out["conforming"]).append(col)
    for col, dist in _persona_cat(persona).items():
        if col not in test_features: continue
        v = str(test_features[col])
        top_vals = set(map(str, (dist.get("top_values") or dist.get("values")
                                 or list(dist)[:3] or [])))
        in_prof = (v in top_vals) if top_vals else True
        out["categorical"][col] = {"value": v, "in_profile": in_prof}
        (out["conforming"] if in_prof else out["deviant"]).append(col)
    return out

def format_deviation_table(dev: dict) -> str:
    lines = ["## DEVIATION TABLE (system-computed vs cluster profile)"]
    for col, d in dev["numeric"].items():
        tag = "DEVIANT" if abs(d["z"]) >= Z_TAU else "ok"
        lines.append(f"  - {col}: value={d['value']}, z={d['z']:+.2f}  [{tag}]")
    for col, d in dev["categorical"].items():
        tag = "ok" if d["in_profile"] else "DEVIANT"
        lines.append(f"  - {col}: '{d['value']}'  [{tag}]")
    if not dev["numeric"] and not dev["categorical"]:
        lines.append("  (no cluster profile available)")
    return "\n".join(lines)


In [14]:
# ── F1-b. TEACHER_TASK 설정 + 프롬프트/파서/rejection 디스패처 (agent/forward 공용) ──
# ★ 단일 노트북: 이 한 줄로 생성 대상 전환. 샘플링/z-table/balance/eval분리/진단은 완전 동일,
#   prompt(prompts.py) + key + rejection 기준만 task로 분기.
import prompts, re as _re, math as _math
from prompts import build_evaluator_user_prompt_v2, build_analyzer_user_prompt_forward_v2
prompts.set_prompt_version(globals().get("STAGE_PROMPT_VER", "v2"))   # S1=v1 / S2-S4=v2
if hasattr(prompts, "set_agent_keys"):
    prompts.set_agent_keys(globals().get("AGENT_KEYS") or "1key")     # S3=3key instruction

EVAL_SAMPLE_N = globals().get("EVAL_SAMPLE_N", 100)
assert TEACHER_TASK in ("agent", "forward")

if TEACHER_TASK == "forward":
    _TEXT_KEY = "reasoning";    prompts.set_mode("forward")
else:
    _TEXT_KEY = "what_missing"; prompts.set_mode("agent")
# ★ stage별 stem — S1/_s1, S3/_3key로 파일 분리 (S2/S4는 기존 이름 유지 = 하위호환)
_STEM = globals().get("STAGE_STEM") or (
    "training_data_forward" if TEACHER_TASK == "forward" else "training_data_mined")
OUT_JSONL_TASK      = f"{OUT_BASE}/{_STEM}.jsonl"
OUT_JSONL_TASK_EVAL = f"{OUT_BASE}/{_STEM}_eval.jsonl"
print(f"■ TEACHER_TASK={TEACHER_TASK}  key={_TEXT_KEY}  →  {OUT_JSONL_TASK}")

# EVALUATOR_TEACHER_RULES = """Ground rules for your JSON:
# - sufficiency: copy the [SYSTEM-COMPUTED] value; never change it.
# - cls:
#     * confident → follow nd (nd≲0.3 NORMAL, nd≳0.55 ABNORMAL).
#     * ambiguous → nd is UNRELIABLE; decide NORMAL vs ABNORMAL by weighing the CLUSTER
#       FIT z-table and the top-k agrees/differs agreement, NOT nd.
# - what_missing assesses RETRIEVAL QUALITY, INDEPENDENT of cls. null when confident;
#   when ambiguous pick the direction by the retrieval situation:
#     * cluster_mismatch — the txn deviates from ITS ASSIGNED persona on a NUMERIC feature
#       ([DEVIANT] in the FIT table) → belongs elsewhere → reroute. Cite the numeric feature
#       with its z (e.g. "cluster_mismatch: purchase_value z=+2.3 above cluster mean").
#       Do NOT invent categorical deviations — categoricals carry no cluster-fit signal here.
#     * neighbor_insufficient — the retrieved neighbors are spread out (RETRIEVAL shows
#       neighbor_spread [spread], not [dense]) or DISAGREE with the txn. Cite 1-2 specific
#       features listed under 'differs' and the neighbor state
#       (e.g. "neighbor_insufficient: source, IP_country differ from neighbors; neighbors spread").
#       This is where categorical differences naturally appear.
# - Never use vague phrases without naming the feature; never restate distance/boundary."""

EVALUATOR_TEACHER_RULES = """Ground rules:
- Output ONLY {"what_missing": "<direction>: <evidence>"}. Never output cls or sufficiency.
- what_missing judges RETRIEVAL QUALITY. Pick the direction by the retrieval situation:
    * cluster_mismatch — the txn deviates from ITS ASSIGNED cluster's persona on a NUMERIC
      feature ([DEVIANT] in the FIT table) → it likely belongs to another cluster → reroute.
      Cite the feature with its z (e.g. "cluster_mismatch: purchase_value z=+2.3 above mean").
      Do NOT invent categorical deviations — categoricals carry no cluster-fit signal here.
    * neighbor_insufficient — neighbors are spread ([spread], not [dense]) or DISAGREE
      (more 'differs' than 'agrees') → too weak to decide → expand.
      Cite 1-2 features under 'differs' + the neighbor state
      (e.g. "neighbor_insufficient: source, IP_country differ from neighbors; neighbors spread").
- Cite 1-2 features FROM THE TABLES ABOVE with values. Never restate distance/boundary."""

# FORWARD_TEACHER_RULES = """Ground rules for your JSON:
# - cls: if confident, follow nd (nd≲0.3→NORMAL, nd≳0.7→ABNORMAL; band may shift);
#   if ambiguous, nd is unreliable — weigh the CLUSTER FIT table + retrieval.
# - reasoning MUST cite features from the CLUSTER FIT table by name, with value/z.
# - Focus on THE SINGLE MOST DECISIVE feature (one clear point beats three vague ones);
#   nd may be secondary evidence.
# - Evidence sources = the CLUSTER FIT table and the top-k agrees/differs line — cite ONLY from those.
# - If ABNORMAL: the decisive feature must be [DEVIANT] in the FIT table or under 'differs',
#   with direction/value (e.g. "purchase_value z=+2.3 far above cluster mean").
# - If NORMAL: cite the 1-2 features with the LOWEST |z| naming their values (each case
#   differs — no generic sentence), or an 'agrees' categorical when relevant.
# - NEVER use vague phrases like "close to cluster range" without naming values.
# - confidence: 0-100 integer reflecting how decisive the evidence is."""

FORWARD_TEACHER_RULES = """Ground rules for your JSON:
- reasoning MUST cite features from the CLUSTER FIT table by name, with value/z.
- Focus on THE SINGLE MOST DECISIVE feature (one clear point beats three vague ones);
  nd may be secondary evidence.
- Evidence sources = the CLUSTER FIT table and the top-k agrees/differs line — cite ONLY from those.
- If ABNORMAL: the decisive feature must be [DEVIANT] in the FIT table or under 'differs',
  with direction/value (e.g. "purchase_value z=+2.3 far above cluster mean").
- If NORMAL: cite the 1-2 features with the LOWEST |z| naming their values (each case
  differs — no generic sentence), or an 'agrees' categorical when relevant.
- NEVER use vague phrases like "close to cluster range" without naming values.
- confidence: 0-100 integer reflecting how decisive the evidence is."""
def build_forward_teacher_prompt(user_v2, gt_label):
    gt = "ABNORMAL" if gt_label == 1 else "NORMAL"
    return (f"{user_v2}\n{FORWARD_TEACHER_RULES}\n\n"
            f"GROUND TRUTH: {gt} — generate reasoning explaining WHY, grounded in the tables above.")

# def build_evaluator_teacher_prompt(user_v2, gt_label):
#     gt = "ABNORMAL" if gt_label == 1 else "NORMAL"
#     return (f"{user_v2}\n{EVALUATOR_TEACHER_RULES}\n\n"
#             f"GROUND TRUTH: {gt} - if evidence is insufficient, give what_missing "
#             f"grounded in the tables above; otherwise null.")

# ── [S3] 3-key teacher rules — teacher가 cls도 evidence로 판단 (GT-blind, 이후 GT-rejection) ──
EVALUATOR_TEACHER_RULES_3KEY = """Ground rules for your JSON:
- sufficiency: copy the [SYSTEM-COMPUTED] value; never change it.
- cls:
    * confident → follow nd (nd≲0.3 NORMAL, nd≳0.55 ABNORMAL).
    * ambiguous → nd is UNRELIABLE; decide NORMAL vs ABNORMAL by weighing the CLUSTER
      FIT z-table and the top-k agrees/differs agreement, NOT nd.
- what_missing assesses RETRIEVAL QUALITY, INDEPENDENT of cls. null when confident;
  when ambiguous pick the direction by the retrieval situation:
    * cluster_mismatch — the txn deviates from ITS ASSIGNED persona on a NUMERIC feature
      ([DEVIANT] in the FIT table) → belongs elsewhere → reroute. Cite the numeric feature
      with its z (e.g. "cluster_mismatch: purchase_value z=+2.3 above cluster mean").
      Do NOT invent categorical deviations — categoricals carry no cluster-fit signal here.
    * neighbor_insufficient — the retrieved neighbors are spread out (RETRIEVAL shows
      neighbor_spread [spread], not [dense]) or DISAGREE with the txn. Cite 1-2 specific
      features listed under 'differs' and the neighbor state
      (e.g. "neighbor_insufficient: source, IP_country differ from neighbors; neighbors spread").
- Never use vague phrases without naming the feature; never restate distance/boundary."""

def build_evaluator_teacher_prompt(user_v2, gt_label):   # gt_label 미사용(blind 공통)
    # S4(1key): teacher는 wm만 생성 (rationalization 차단)
    # S3(3key): teacher가 cls도 evidence로 생성 → accept_and_build에서 GT-rejection
    _rules = (EVALUATOR_TEACHER_RULES_3KEY if globals().get("AGENT_KEYS") == "3key"
              else EVALUATOR_TEACHER_RULES)
    return f"{user_v2}\n{_rules}"

# ── [S1] v1 forward teacher — GT-conditioned + teacher 자체 confidence (rationalization 재현) ──
def build_forward_teacher_prompt_v1(user_v1, gt_label):
    gt = "ABNORMAL" if gt_label == 1 else "NORMAL"
    return (f"{user_v1}\n\nGROUND TRUTH: {gt}\n\n"
            f"Generate reasoning explaining WHY this is {gt}. "
            "Use normalized distance as key signal; cite specific distance and feature values. "
            "Confidence: strong evidence 80-95, moderate 60-80, weak 50-70.")

# ===== 파서 (task별) =====
def _parse_forward(text):
    t = text
    if "<think>" in t:
        t = _re.sub(r"<think>.*?</think>", "", t, flags=_re.DOTALL).strip()
    if "```" in t:
        t = t.split("```json")[-1].split("```")[0] if "```json" in t else t.split("```")[1]
    if "{" in t: t = "{" + t.split("{", 1)[1]
    if "}" in t: t = t.rsplit("}", 1)[0] + "}"
    p = json.loads(t); cls = str(p.get("cls", p.get("classification", ""))).upper()
    if cls not in ("NORMAL", "ABNORMAL"): raise ValueError(f"cls={cls}")
    conf = max(0, min(100, int(float(p.get("confidence", 50)))))
    reasoning = str(p.get("reasoning", "")).strip()
    if not reasoning: raise ValueError("empty reasoning")
    return {"cls": cls, "confidence": conf, "reasoning": reasoning[:300]}

def _parse_eval2(text):
    """v2 evaluator 1-key {what_missing} 파서. cls/sufficiency는 코드가 결정."""
    t = text.split("</think>")[-1] if "</think>" in text else text
    if "```" in t:
        t = t.split("```json")[-1].split("```")[0] if "```json" in t else t.split("```")[1]
    if "{" in t and "}" in t: t = t[t.index("{"): t.rindex("}") + 1]
    else: raise ValueError("no JSON")
    p = json.loads(t)
    wm = p.get("what_missing")
    wm = wm.strip() if isinstance(wm, str) and wm.strip().lower() not in ("null", "none", "") else None
    return {"what_missing": wm}

def _parse_eval3(text):
    """[S3] v2 evaluator 3-key 파서 — cls는 teacher evidence, sufficiency는 코드가 재결정."""
    t = text.split("</think>")[-1] if "</think>" in text else text
    if "```" in t:
        t = t.split("```json")[-1].split("```")[0] if "```json" in t else t.split("```")[1]
    if "{" in t and "}" in t: t = t[t.index("{"): t.rindex("}") + 1]
    else: raise ValueError("no JSON")
    p = json.loads(t)
    cls = str(p.get("cls", p.get("classification", ""))).upper()
    if cls not in ("NORMAL", "ABNORMAL"): raise ValueError(f"cls={cls}")
    wm = p.get("what_missing")
    wm = wm.strip() if isinstance(wm, str) and wm.strip().lower() not in ("null", "none", "") else None
    return {"cls": cls, "what_missing": wm}

# ===== grounded rejection (task별) =====
def _cited_features(text, dev):
    r = (text or "").lower()
    return [c for c in list(dev["numeric"]) + list(dev["categorical"]) if c.lower() in r]

def grounded_ok_forward(cls, reasoning, dev):
    cited = _cited_features(reasoning, dev)
    if not cited: return False
    if cls == "ABNORMAL": return any(c in dev["deviant"] for c in cited)
    return any(c in dev["conforming"] for c in cited)     # B-2 대칭 검증

_WM_DIR = ("cluster_mismatch", "neighbor_insufficient")
def grounded_ok_evaluator(cls, wm, dev):
    if wm is None: return True                            # confident=null 허용
    low = wm.lower()
    if not any(low.startswith(d) for d in _WM_DIR): return False
    if "distance" in low or "boundary" in low: return False   # 거리 재진술 차단
    cited = [c for c in list(dev["numeric"]) + list(dev["categorical"]) if c.lower() in low]
    if not cited: return False
    # return any(c in dev["deviant"] for c in cited) if low.startswith("cluster_mismatch") else True
    #변경
    direction = "cluster_mismatch" if low.startswith("cluster_mismatch") else "neighbor_insufficient"
    cites_dev = any(c in dev["deviant"] for c in cited)
    if cls == "NORMAL"   and (direction == "cluster_mismatch" or cites_dev): return False  # ★ 정합
    if cls == "ABNORMAL" and not cites_dev: return False
    return True

def _parse_retr(u):
    _i = (u or "").find("## RETRIEVAL")
    r  = u[_i:] if _i >= 0 else ""              # ★ instruction 제외, RETRIEVAL 블록만
    m = re.search(r"agrees:\s*([^|]*)\|\s*differs:\s*([^\n]*)", r)
    _n = lambda s: (0 if (not s.strip() or s.strip().startswith("(none)"))
                    else len([x for x in s.split(",") if x.strip()]))
    return {"spread": "[spread]" in r,          # ★ r에서만
            "agree": _n(m.group(1)) if m else 0, "differ": _n(m.group(2)) if m else 0}

def grounded_wm_direction(wm, dev, retr):
    low = wm.lower()
    if not any(low.startswith(d) for d in _WM_DIR): return False
    if "distance" in low or "boundary" in low: return False
    if low.startswith("cluster_mismatch"):
        cited = [c for c in list(dev["numeric"]) + list(dev["categorical"]) if c.lower() in low]
        return any(c in dev["deviant"] for c in cited)             # persona 이탈
    return retr["spread"] or (retr["differ"] > 0 and retr["differ"] >= retr["agree"])

def relabel_confidence(nd):   # forward 전용
    return int(round(50 + 50 * _math.tanh(abs(nd - 0.5) * 4)))

# ===== task 디스패처 (stage-aware) =====
if TEACHER_TASK == "forward":
    if globals().get("STAGE_PROMPT_VER", "v2") == "v1":               # ★ S1: v1 프롬프트
        build_task_user_prompt    = lambda tf,cid,oi,dd,per: prompts.build_analyzer_user_prompt_forward(
            tf, cid, oi, dd, per)
        build_task_teacher_prompt = build_forward_teacher_prompt_v1
    else:                                                             # S2: v2 프롬프트
        build_task_user_prompt    = lambda tf,cid,oi,dd,per: build_analyzer_user_prompt_forward_v2(
            test_features=tf, cluster_id=cid, top5_orig_indices=oi[:3], distances=dd[:3], persona=per)
        build_task_teacher_prompt = build_forward_teacher_prompt
    _parse_task, grounded_ok_task = _parse_forward, grounded_ok_forward
else:
    build_task_user_prompt    = lambda tf,cid,oi,dd,per: build_evaluator_user_prompt_v2(
        tf, cid, oi[:3], dd[:3], per)
    build_task_teacher_prompt = build_evaluator_teacher_prompt
    # ★ S3: teacher가 cls도 출력 → 3-key 파서 / S4: wm-only 파서
    _parse_task = _parse_eval3 if globals().get("AGENT_KEYS") == "3key" else _parse_eval2
    grounded_ok_task = grounded_ok_evaluator

def _hf_batch_infer_task(prompts_, batch_size=None):
    import torch
    from tqdm.auto import tqdm
    bs = batch_size or HF_BATCH_SIZE
    results = [None] * len(prompts_)
    chats = [[{"role":"system","content":SYSTEM_PROMPT},
              {"role":"user","content":p + "\n\nIMPORTANT: Respond ONLY with valid JSON, no markdown."}]
             for p in prompts_]
    for s in tqdm(range(0, len(chats), bs), desc=f"{TEACHER_TASK} teacher 배치"):
        texts = [hf_tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True)
                 for c in chats[s:s+bs]]
        enc = hf_tokenizer(texts, return_tensors="pt", padding=True,
                           truncation=True, max_length=2048).to(hf_model.device)
        with torch.no_grad():
            out = hf_model.generate(**enc, max_new_tokens=160, do_sample=False,
                                    pad_token_id=hf_tokenizer.pad_token_id)
        for j in range(len(texts)):
            try:
                results[s+j] = _parse_task(hf_tokenizer.decode(out[j][enc["input_ids"].shape[1]:],
                                                               skip_special_tokens=True))
            except Exception:
                results[s+j] = None
    return results


■ TEACHER_TASK=forward  key=reasoning  →  /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_forward_s1.jsonl


In [15]:
# ── [Agent 3-key] deterministic cls/sufficiency + coupling 강제 (local_analyzer 정합) ──
#   cls·sufficiency는 nd로 결정(모델 자유도 아님). teacher는 what_missing만 생성.
#   confident → wm=null / ambiguous → grounded wm 필수.  (값은 local_analyzer env와 동기화)
SUF_LOW, SUF_HIGH    = 0.3, 0.55        # local_analyzer SUFFICIENCY_BOUNDARY_LOW/HIGH
CLS_ND_THRESHOLD_GEN = 0.5              # local_analyzer CLS_ND_THRESHOLD
def _suf_from_nd(nd): return "ambiguous" if (SUF_LOW <= nd < SUF_HIGH) else "confident"
def _cls_from_nd(nd): return "ABNORMAL" if nd >= CLS_ND_THRESHOLD_GEN else "NORMAL"
def accept_and_build(o, m, is_eval=False):
    """parsed teacher output o + meta m → (assistant_dict, dedup_text) 채택 / (None, reason) 기각.
    is_eval=True : rejection 전부 해제 (teacher가 맞힌 것만 남는 편향 제거).
    REJECTION_ON=False (S1) : train도 필터 해제 — parse 실패만 기각, confidence는 teacher 원본.
    AGENT_KEYS="3key" (S3)  : ambiguous cls = teacher evidence → GT-rejection.
    AGENT_KEYS="1key" (S4)  : cls/sufficiency 전부 nd 결정론, teacher는 wm만."""
    if o is None: return None, "fail"
    nd = float(m["top1_l2_dist_norm"])
    _rej = globals().get("REJECTION_ON", True) and not is_eval   # S1: train도 무필터

    if TEACHER_TASK == "forward":
        if _rej:
            if (1 if o["cls"] == "ABNORMAL" else 0) != m["gt_label"]:          # cls-GT
                return None, "gtmiss"
            if not grounded_ok_forward(o["cls"], o.get("reasoning"), m["_dev"]):
                return None, "ungrounded"
        if globals().get("CONF_RELABEL", True) is not False:                    # S2: 라벨 함수
            o["confidence"] = relabel_confidence(nd)
        return o, o.get("reasoning")                                            # S1: teacher conf 유지

    cls_det, suf_det = _cls_from_nd(nd), _suf_from_nd(nd)
    if suf_det == "confident":                          # confident: cls=nd, wm=null (공통)
        return {"cls": cls_det, "sufficiency": "confident", "what_missing": None}, None

    wm = o.get("what_missing")
    if globals().get("AGENT_KEYS") == "3key":           # ★ S3: ambiguous cls = teacher evidence
        cls_amb = o.get("cls", cls_det)
        if is_eval:                                     # eval: 무필터, teacher 판정 보존
            return {"cls": cls_amb, "sufficiency": "ambiguous", "what_missing": wm}, wm
        if _rej:
            if (1 if cls_amb == "ABNORMAL" else 0) != m["gt_label"]:
                return None, "gtmiss"                   # evidence cls의 GT-rejection
            if not wm or not grounded_wm_direction(wm, m["_dev"], m["_retr"]):
                return None, "ungrounded"
        return {"cls": cls_amb, "sufficiency": "ambiguous", "what_missing": wm}, wm

    # S4 (1key): cls도 nd 결정론
    if is_eval:
        return {"cls": cls_det, "sufficiency": "ambiguous", "what_missing": wm}, wm
    if _rej and (not wm or not grounded_wm_direction(wm, m["_dev"], m["_retr"])):
        return None, "ungrounded"
    return {"cls": cls_det, "sufficiency": "ambiguous", "what_missing": wm}, wm

    # # ── agent ──
    # cls_det, suf_det = _cls_from_nd(nd), _suf_from_nd(nd)
    # if suf_det == "confident":                                     # confident: cls=nd, wm=null
    #     return {"cls": cls_det, "sufficiency": "confident", "what_missing": None}, None

    # wm = o.get("what_missing")
    # if is_eval:
    #     # ★ eval: 필터 없음. train 분포와 동일하게 ambiguous cls는 GT
    #     #   (train은 cls-GT 필터를 통과한 것만 남으므로 결과적으로 cls==GT)
    #     return {"cls": "ABNORMAL" if m["gt_label"] == 1 else "NORMAL",
    #             "sufficiency": "ambiguous", "what_missing": wm}, wm

    # # ambiguous(train): cls는 evidence(teacher blind) → GT로 rejection
    # if (1 if o["cls"] == "ABNORMAL" else 0) != m["gt_label"]:
    #     return None, "cls_wrong"
    # if not wm or not grounded_wm_direction(wm, m["_dev"], m["_retr"]):
    #     return None, "ungrounded"
    # return {"cls": o["cls"], "sufficiency": "ambiguous", "what_missing": wm}, wm   # ★ wm 복원

## Section F2: 샘플링 — nd bin intuition 반영
nd bin 분석(0.4~0.6 acc 최저)에 따라 boundary 정의를 **[0.4, 0.6) rank**로 이동,
NB01 교훈(boundary 편중 → 학습 신호 왜곡)대로 easy와 균형 배분. mining은 소량 유지.

In [16]:
def stratified_sample(sampled: List[Tuple[int, str]], n: int) -> List[Tuple[int, str]]:
    """category별 균등 비율로 n개 추출."""
    from collections import defaultdict
    buckets = defaultdict(list)
    for item in sampled:
        buckets[item[1]].append(item)

    n_cats  = len(buckets)
    per_cat = max(1, n // n_cats)
    result  = []
    rng     = random.Random(42)
    for cat, items in buckets.items():
        result += rng.sample(items, min(per_cat, len(items)))

    rng.shuffle(result)
    return result[:n]

def load_mined_as_sampled(
    parquet_path: str,
    target_size: int = None,
    exclude_last_pct: float = 0.0,
    easy_pool: List[Tuple[int, str]] = None,   # ✅ distance set의 easy 차용 (혼합 모드)
    mining_ratio: float = None,                 # ✅ easy:mining 비율 (1.0=전부 mining)
) -> List[Tuple[int, str]]:
    """
    Mined set: mining(type1+type2, type3 제외) M개 + easy (N-M)개.
    - mining_ratio=1.0 → 전부 mining, easy 보충 없음 (mining_only 모드)
    - mining_ratio<1.0 → easy_pool(=distance set)에서 (N-M)개 차용 → ablation 통제
    """
    if target_size is None:
        target_size = N_NORMAL_TARGET + N_ABNORMAL_TARGET
    if mining_ratio is None:
        mining_ratio = MINING_RATIO

    mined_df = pd.read_parquet(parquet_path)

    # ── ✅ type3 제외 ──
    mined_df = mined_df[mined_df["mining_type"] != "type3_distance_band"].copy()
    print(f"  [mined parquet] type3 제외 후 {len(mined_df)}행")
    print(f"  mining_type:\n{mined_df['mining_type'].value_counts().to_string()}")
    print(f"  gt_label: normal={int((mined_df['gt_label']==0).sum())}  "
          f"fraud={int((mined_df['gt_label']==1).sum())}")

    type_map = {
        "type1_misrouted":     "hard_negative_type1",
        "type2_cross_cluster": "hard_negative_type2",
    }
    mined_all = [
        (int(row["test_idx"]), type_map.get(row["mining_type"], "hard_negative"))
        for _, row in mined_df.iterrows()
        if row["mining_type"] in type_map           # type3 잔여 방어
    ]

    # ── ✅ mining 목표 개수 = target * mining_ratio ──
    n_mining_target = max(1, int(round(target_size * mining_ratio)))

    # mining이 목표보다 많으면 stratified(type별 균등)로 축소 — GT 50:50 강제 안 함(자연분포)
    if len(mined_all) > n_mining_target:
        mining_sel = stratified_sample(mined_all, n_mining_target)
    else:
        mining_sel = mined_all
    n_mining_actual = len(mining_sel)
    n_easy_actual   = max(0, target_size - n_mining_actual)   # ✅ 음수 방지

    print(f"\n  [구성] mining={n_mining_actual} (목표 {n_mining_target})  "
          f"easy={n_easy_actual}")

    # ── ✅ easy 보충: 필요할 때만 (mining_ratio<1.0인 경우). 0이면 가드/차용 모두 skip ──
    if n_easy_actual == 0:
        easy_sel = []                                          # mining_only → easy 불필요
        n_en = n_ea = 0
    else:
        if easy_pool is None:
            raise ValueError("easy 보충 필요(mining_ratio<1.0)한데 easy_pool 없음 — dispatch에서 전달하세요")
        mined_idx_set = set(i for i, _ in mining_sel)
        # easy_pool을 label별로 분리 (50:50 보충용)
        pool_normal   = [(i, c) for i, c in easy_pool if c == "easy_normal"   and i not in mined_idx_set]
        pool_abnormal = [(i, c) for i, c in easy_pool if c == "easy_abnormal" and i not in mined_idx_set]
        n_en = min(n_easy_actual // 2, len(pool_normal))
        n_ea = min(n_easy_actual - n_easy_actual // 2, len(pool_abnormal))
        easy_sel = pool_normal[:n_en] + pool_abnormal[:n_ea]

    result = mining_sel + easy_sel
    random.Random(42).shuffle(result)

    print(f"  [최종] total={len(result)}  mining={n_mining_actual}  "
          f"easy(normal={n_en}, abnormal={n_ea})  (target={target_size})")
    return result

In [17]:
# ── F2. 배분: EASY 0.4 / BOUNDARY(0.4~0.6) 0.4 / MINING 0.2 ──
FWD_TARGET_SIZE   = 1200
FWD_OVERSAMPLE    = 2.5          # rejection(cls-GT + grounded) 감안
FWD_EASY_FRAC, FWD_BOUNDARY_FRAC, FWD_MINING_FRAC = 0.2, 0.5, 0.3

_gen_target = int(FWD_TARGET_SIZE * FWD_OVERSAMPLE)
n_easy, n_boundary = int(_gen_target*FWD_EASY_FRAC), int(_gen_target*FWD_BOUNDARY_FRAC)
n_mining = _gen_target - n_easy - n_boundary

# boundary: 최난 구간 [0.4, 0.6) — NB01 sample_boundary_distance 재사용 (rank 기준)
_boundary = sample_boundary_distance(low=0.3, high=0.55, target=n_boundary,
    exclude_first_pct=TRAIN_RATIO, exclude_last_pct=EVAL_RATIO)
_easy_n = sample_boundary_distance(low=0.0, high=0.3, target=n_easy//2,
    exclude_first_pct=TRAIN_RATIO, exclude_last_pct=EVAL_RATIO)
_easy_a = sample_boundary_distance(low=0.55, high=1.0, target=n_easy - n_easy//2,
    exclude_first_pct=TRAIN_RATIO, exclude_last_pct=EVAL_RATIO)
_mining = load_mined_as_sampled(MINED_PARQUET_PATH, target_size=n_mining,
    exclude_last_pct=EVAL_RATIO, easy_pool=None, mining_ratio=1.0) if FWD_MINING_FRAC else []

fwd_sampled = _mining + _boundary + _easy_n + _easy_a
print(f"[F2] mining={len(_mining)} boundary={len(_boundary)} "
      f"easy={len(_easy_n)+len(_easy_a)} → 총 {len(fwd_sampled)} (목표 채택 {FWD_TARGET_SIZE})")

# ── mining_type 조회 맵 (direction GT) — boundary/easy 샘플에도 적용 ──
import pandas as pd
_mdf_all = pd.read_parquet(MINED_PARQUET_PATH)
MINING_TYPE = {int(r["test_idx"]): str(r["mining_type"]) for _, r in _mdf_all.iterrows()}
get_mining_type = lambda ti: MINING_TYPE.get(int(ti), "clean")
print(f"[F2] mining_type 맵 {len(MINING_TYPE)}건 | "
      f"컬럼={_mdf_all.columns.tolist()}")     # ★ router 배정 컬럼 있는지 함께 확인
# ── router 배정 맵 (misroute를 프롬프트에서 관측 가능하게) ──
ROUTER_CLUSTER = {int(r["test_idx"]): int(r["predicted_cluster"]) for _, r in _mdf_all.iterrows()}
get_router_cluster = lambda ti: ROUTER_CLUSTER.get(int(ti), int(gt_clusters[ti]))
_nm = sum(1 for ti, c in ROUTER_CLUSTER.items() if c != int(gt_clusters[ti]))
print(f"[F2] router 맵 {len(ROUTER_CLUSTER)}건 | predicted≠gt: {_nm} ({_nm/max(len(ROUTER_CLUSTER),1):.1%})")


  [boundary 0.3~0.55] 11399개 중 1500개 선택 (ABNORMAL 404)
  [boundary 0.0~0.3] 2180개 중 300개 선택 (ABNORMAL 1)
  [boundary 0.55~1.0] 9063개 중 300개 선택 (ABNORMAL 271)
  [mined parquet] type3 제외 후 1794행
  mining_type:
mining_type
type2_cross_cluster    1263
type1_misrouted         531
  gt_label: normal=837  fraud=957

  [구성] mining=900 (목표 900)  easy=0
  [최종] total=900  mining=900  easy(normal=0, abnormal=0)  (target=900)
[F2] mining=900 boundary=1500 easy=600 → 총 3000 (목표 채택 1200)
[F2] mining_type 맵 1922건 | 컬럼=['test_idx', 'pred_pos', 'gt_label', 'gt_cluster_id', 'top5_indices', 'predicted_cluster', 'top_k_clusters', 'match_type', 'retrieved_original_indices', 'top1_distance', 'feat_age', 'feat_sex', 'feat_source', 'feat_browser', 'mining_type', 'experiment_case', 'percolate_version', 'neighbor_cluster_ids', 'cross_cluster_ids', 'n_cross_cluster_neighbors', 'dist_p75', 'dist_p90']
[F2] router 맵 1922건 | predicted≠gt: 531 (27.6%)


## Section F3: 생성 + 3중 Rejection + Train/Eval 분리
Rejection 축 3개 — 전부 **검증 가능한 신호** 기반 (heuristic 금지어 아님):
1. **cls-GT** (기존): teacher cls ≠ GT → 제외
2. **Grounded-z**: cls=ABNORMAL인데 reasoning이 인용한 feature 중 실제 [DEVIANT]가
   0개면 제외 / cls=NORMAL인데 존재하지 않는 feature를 인용하면 제외
3. **confidence 재라벨** (reject 대신 교정): teacher confidence의 평균회귀를 라벨
   단계에서 차단 — `conf = 50 + 50·tanh(|z_nd|)` 형태로 nd 극단성 기반 재산출
   (GT 일치가 전제이므로 verifiable — "확신 라벨을 거리 캘리브레이션으로 가르치기")

In [18]:
# ── F3. 생성 + Rejection (task 공용, agent=3-key {cls,sufficiency,what_missing}) ──
import re as _re

def _save_jsonl(records, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records: f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  💾 저장: {path}  ({len(records)}개, {os.path.getsize(path)/1024:.0f} KB)")

def _balance_and_trim(records, target_n, tag=""):
    import random as _r
    norm = [x for x in records if x['metadata']['gt_label'] == 0]
    ab   = [x for x in records if x['metadata']['gt_label'] == 1]
    n_each = min(len(norm), len(ab), target_n // 2)
    _r.Random(42).shuffle(norm); _r.Random(42).shuffle(ab)
    out = norm[:n_each] + ab[:n_each]; _r.Random(42).shuffle(out)
    print(f"  [balance:{tag}] 통과 {len(records)} (N={len(norm)}, AB={len(ab)}) "
          f"→ 채택 {len(out)} (N={n_each}, AB={n_each})")
    return out

def _inject_suf(user_v2, nd):
    """agent det: 서빙(local_analyzer)과 동일하게 sufficiency를 가변 후미에 주입 → train-serve 정합."""
    if TEACHER_TASK == "agent":
        return user_v2 + f"\n\n[SYSTEM-COMPUTED] retrieval_sufficiency = {_suf_from_nd(nd).upper()}"
    return user_v2

# ── 생성 루프 ──
fwd_prompts, fwd_users, fwd_metas = [], [], []
for test_idx, category in fwd_sampled:
    gt_label   = int(labels[test_idx]); cluster_id = int(gt_clusters[test_idx])
    cluster_id = get_router_cluster(test_idx)          # ★ GT → router 배정 (토큰 증가 0)
    top5_oi    = [int(x) for x in top5_indices[test_idx]]; persona = CLUSTER_PERSONAS.get(cluster_id)
    tf         = extract_test_features(test_idx); dists = compute_l2_distances(test_emb[test_idx], top5_oi)
    dev        = build_deviation_table(tf, persona)
    _nd        = normalize_dist(dists[0])
    _user_v2   = _inject_suf(build_task_user_prompt(tf, cluster_id, top5_oi, dists, persona), _nd)
    fwd_users.append(_user_v2)
    fwd_prompts.append(build_task_teacher_prompt(_user_v2, gt_label))
    fwd_metas.append({"test_idx": int(test_idx), "gt_label": gt_label, "category": category,
                      "cluster_id": cluster_id, "top1_l2_distance": float(dists[0]),
                      "top1_l2_dist_norm": _nd, "_dev": dev, "_retr": _parse_retr(_user_v2),
                      "_mining_type": get_mining_type(test_idx),
                      "_misroute":   int(get_mining_type(test_idx) == "type1_misrouted"),
                      "_retr_error": int(get_mining_type(test_idx) == "type2_cross_cluster"),})

# ── [사전 검증] 프롬프트 무결성 (버그② 재발 방지: GPU 소모 전에 실패) ──
_u0 = fwd_users[0]
if globals().get("STAGE_PROMPT_VER", "v2") == "v2":
    assert "## RETRIEVAL" in _u0, "RETRIEVAL 블록 없음"
    assert "## CLUSTER" in _u0,   "CLUSTER FIT 블록 없음"
    _n_agd = sum("agrees:" in u for u in fwd_users)
    print(f"[검증] agrees/differs 라인 존재율: {_n_agd}/{len(fwd_users)} = {_n_agd/len(fwd_users):.1%}")
    assert _n_agd == len(fwd_users), (
        "★ 버그② 재발 — topk_feats 미전달. prompts.build_*_v2의 "
        "coreset_feature_map 조회(list comprehension) 확인")
else:
    print("[검증] v1 프롬프트(S1) — RETRIEVAL/agrees 무결성 검증 생략 (v2 전용 블록 부재가 정상)")
# if TEACHER_TASK == "agent":
#     _n_suf = sum(u.count("[SYSTEM-COMPUTED]") for u in fwd_users)
#     assert _n_suf == len(fwd_users), f"sufficiency 주입 이상 (중복/누락): {_n_suf}"
if TEACHER_TASK == "agent":
    # ★ instruction에도 '[SYSTEM-COMPUTED]' 문구가 있으므로, 주입 라인 고유 토큰으로 카운트
    _n_suf = sum(u.count("retrieval_sufficiency =") for u in fwd_users)
    assert _n_suf == len(fwd_users), f"sufficiency 주입 이상 (중복/누락): {_n_suf}"
print("[검증] ✅ 프롬프트 무결성 통과 →", repr(_u0[_u0.find("## RETRIEVAL"):][:150]))

outs = _hf_batch_infer_task(fwd_prompts)

fwd_records = []; n_fail = n_gtmiss = n_ungrounded = n_dup = 0
_dup_seen = {}; _DUP_CAP = max(3, int(FWD_TARGET_SIZE * 0.02))
_dup_key = lambda t: _re.sub(r"[^a-z0-9 ]", "", (t or "").lower())[:40]
for o, m, up in zip(outs, fwd_metas, fwd_users):
    rec, tag = accept_and_build(o, m)
    if rec is None:
        if tag == "fail": n_fail += 1
        elif tag == "gtmiss": n_gtmiss += 1
        else: n_ungrounded += 1
        continue
    if tag and globals().get("REJECTION_ON", True):            # dedup (confident-null 제외, S1은 해제)
        _k = _dup_key(tag); _dup_seen[_k] = _dup_seen.get(_k, 0) + 1
        if _dup_seen[_k] > _DUP_CAP: n_dup += 1; continue
    meta = {k: v for k, v in m.items() if k not in ("_dev", "_retr")}   # ★ _retr도 제거
    fwd_records.append({"system": SYSTEM_PROMPT, "user": up,
                        "assistant": json.dumps(rec, ensure_ascii=False), "metadata": meta})
    # meta = {k: v for k, v in m.items() if k != "_dev"}
    # fwd_records.append({"system": SYSTEM_PROMPT, "user": up,
    #                     "assistant": json.dumps(rec, ensure_ascii=False), "metadata": meta})
print(f"[F3] 생성 {len(outs)} → 채택 {len(fwd_records)} "
      f"(fail {n_fail} / gtmiss {n_gtmiss} / ungrounded {n_ungrounded} / dup {n_dup})")

# ── ★ Fig 1(Rejection funnel Sankey) 소스: 생성/기각 카운트 저장 → NB03이 로드 ──
_gen_stats = {"stage": globals().get("EXPERIMENT_STAGE", "?"), "task": TEACHER_TASK,
              "generated": len(outs), "fail": n_fail, "gtmiss": n_gtmiss,
              "ungrounded": n_ungrounded, "dup": n_dup, "accepted": len(fwd_records)}
GEN_STATS_PATH = f"{OUT_BASE}/{_STEM}_gen_stats.json"
os.makedirs(OUT_BASE, exist_ok=True)
with open(GEN_STATS_PATH, "w") as _f: json.dump(_gen_stats, _f)
print(f"  💾 gen_stats 저장: {GEN_STATS_PATH}")

# fwd_records = _balance_and_trim(fwd_records, FWD_TARGET_SIZE, TEACHER_TASK)


[검증] v1 프롬프트(S1) — RETRIEVAL/agrees 무결성 검증 생략 (v2 전용 블록 부재가 정상)
[검증] ✅ 프롬프트 무결성 통과 → '}'


forward teacher 배치:   0%|          | 0/60 [00:00<?, ?it/s]

[F3] 생성 3000 → 채택 3000 (fail 0 / gtmiss 0 / ungrounded 0 / dup 0)
  💾 gen_stats 저장: /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_forward_s1_gen_stats.json


In [19]:
# ── F3-c. Eval 생성 + train/eval 저장 (task 공용, TRAIN과 동일 정합) ──
n_total = len(labels); _eval_start = int(n_total * (1 - EVAL_RATIO))
_pN  = [i for i in range(_eval_start, n_total) if int(labels[i]) == 0]
_pAB = [i for i in range(_eval_start, n_total) if int(labels[i]) == 1]
_n_each = min(len(_pN), len(_pAB))
if EVAL_SAMPLE_N is not None: _n_each = min(_n_each, EVAL_SAMPLE_N // 2)
import random as _r
_rng = _r.Random(42)
_eval_idx = _rng.sample(_pN, _n_each) + _rng.sample(_pAB, _n_each); _rng.shuffle(_eval_idx)
print(f"[F3-c] eval 구간 [{_eval_start},{n_total}) 풀 N={len(_pN)}/AB={len(_pAB)} "
      f"→ 선택 {len(_eval_idx)} (EVAL_SAMPLE_N={EVAL_SAMPLE_N})")

ev_prompts, ev_users, ev_metas = [], [], []
for ti in _eval_idx:
    gtl = int(labels[ti]); cid = get_router_cluster(ti); oi = [int(x) for x in top5_indices[ti]]
    per = CLUSTER_PERSONAS.get(cid); tf = extract_test_features(ti)
    dd  = compute_l2_distances(test_emb[ti], oi); dev = build_deviation_table(tf, per)
    _nd = normalize_dist(dd[0])
    _u  = _inject_suf(build_task_user_prompt(tf, cid, oi, dd, per), _nd)     # ★ TRAIN과 동일 suf 주입
    ev_users.append(_u); ev_prompts.append(build_task_teacher_prompt(_u, gtl))
    ev_metas.append({
        "test_idx": int(ti),
        "gt_label": gtl,
        "cluster_id": cid,
        "top1_l2_distance": float(dd[0]),
        "top1_l2_dist_norm": _nd,
        "_dev": dev,
        "_retr": _parse_retr(_u),

        "_mining_type": f"eval_{get_mining_type(ti)}",
        "_misroute": int(get_mining_type(ti) == "type1_misrouted"),
        "_retr_error": int(get_mining_type(ti) == "type2_cross_cluster"),
    })                  # ★ _retr 추가

# ── [사전 검증] 프롬프트 무결성 (버그② 재발 방지: GPU 소모 전에 실패) ──
if globals().get("STAGE_PROMPT_VER", "v2") == "v2":
    assert all("agrees:" in u for u in ev_users)

ev_outs = _hf_batch_infer_task(ev_prompts)
eval_recs = []
for o, m, up in zip(ev_outs, ev_metas, ev_users):
    rec, _tag = accept_and_build(o, m, is_eval=True)
    if rec is None: continue
    meta = {k: v for k, v in m.items() if k not in ("_dev", "_retr")}        # ★ 임시 키 제거
    eval_recs.append({"system": SYSTEM_PROMPT, "user": up,
                      "assistant": json.dumps(rec, ensure_ascii=False), "metadata": meta})

train_recs = [r for r in fwd_records if r["metadata"]["test_idx"] < _eval_start]   # 누수 차단
_save_jsonl(train_recs, OUT_JSONL_TASK)
_save_jsonl(eval_recs,  OUT_JSONL_TASK_EVAL)
print(f"[F3-c] train={len(train_recs)} eval={len(eval_recs)} "
      f"(N={sum(r['metadata']['gt_label']==0 for r in eval_recs)}, "
      f"AB={sum(r['metadata']['gt_label']==1 for r in eval_recs)})")

# ── ★ Fig 1 소스: 최종 train/eval 카운트 추가 저장 ──
if "_gen_stats" in dir():
    _gen_stats.update({"train_final": len(train_recs), "eval_final": len(eval_recs)})
    with open(GEN_STATS_PATH, "w") as _f: json.dump(_gen_stats, _f)
    print(f"  💾 gen_stats 갱신(train_final/eval_final): {GEN_STATS_PATH}")

[F3-c] eval 구간 [25471,28302) 풀 N=1441/AB=1390 → 선택 300 (EVAL_SAMPLE_N=300)


forward teacher 배치:   0%|          | 0/6 [00:00<?, ?it/s]

  💾 저장: /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_forward_s1.jsonl  (3000개, 9175 KB)
  💾 저장: /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_forward_s1_eval.jsonl  (300개, 909 KB)
[F3-c] train=3000 eval=300 (N=150, AB=150)
  💾 gen_stats 갱신(train_final/eval_final): /content/gcs_mount/tree-search/analyzer_ft/pca_64_k100/hf_0713/training_data_forward_s1_gen_stats.json


## Section F4: 품질 진단 — collapse/편향 사전 검출 (배포 게이트)

In [ ]:
# ── F4. 품질 진단 (forward=reasoning+conf / agent=3-key sufficiency+coupling) ──
import json, re
import numpy as np
from collections import Counter
if "train_recs" not in dir():
    train_recs = [json.loads(l) for l in open(OUT_JSONL_TASK, encoding="utf-8")]

def quality_report(records):
    parsed = [json.loads(r["assistant"]) for r in records]
    allt = [p.get(_TEXT_KEY) for p in parsed]
    ts = [(t or "").strip().lower() for t in allt if t]
    c = Counter(ts); n = len(ts)
    print(f"{_TEXT_KEY} n={n} (non-null)  고유율={len(c)/max(n,1):.1%}  "
          f"top1={c.most_common(1)[0][1]/max(n,1) if c else 0:.1%}")
    print("최빈 3개:", [(t[:55], k) for t, k in c.most_common(3)])
    if TEACHER_TASK == "forward":
        confs = np.array([p["confidence"] for p in parsed])
        nds   = np.array([r["metadata"]["top1_l2_dist_norm"] for r in records])
        print(f"confidence: mean={confs.mean():.1f} std={confs.std():.1f} (std<10이면 평균회귀 위험)")
        print(f"confidence ↔ |nd−0.5| 상관: {np.corrcoef(confs, np.abs(nds-0.5))[0,1]:.3f}")
    else:
        sufs = Counter(p.get("sufficiency") for p in parsed)
        n_conf, n_amb = sufs.get("confident", 0), sufs.get("ambiguous", 0)
        conf_null = sum(1 for p in parsed if p.get("sufficiency")=="confident" and p.get("what_missing") is None)
        amb_wm    = sum(1 for p in parsed if p.get("sufficiency")=="ambiguous" and p.get("what_missing"))
        bad = sum(1 for p in parsed
                  if (p.get("sufficiency")=="confident") != (p.get("what_missing") is None))
        _DIR = re.compile(r"^(cluster_mismatch|neighbor_insufficient)\s*:")
        filled = [t for t in allt if t]; n_dir = sum(bool(_DIR.match(t)) for t in filled)
        print(f"sufficiency 분포: {dict(sufs)}")
        print(f"coupling(=confident⟺null) 위반: {bad}/{len(parsed)}  "
              f"[confident&null {conf_null}/{n_conf}, ambiguous&wm {amb_wm}/{n_amb}]")
        print(f"wm 형식 준수율(direction): {n_dir}/{len(filled)}={n_dir/max(len(filled),1):.1%}  "
              f"분포={dict(Counter(_DIR.match(t).group(1) for t in filled if _DIR.match(t)))}")

quality_report(train_recs)


## Section F5: 심화 진단 (A-1 cls별 / A-2 confidence calibration / A-3 soft-cap)
재생성 없이 기존 jsonl로 실행 — B-1/2/3 반영 재생성 전후 비교 지표로도 사용.

In [21]:
# ============================================================
# [Section F5] 심화 진단 (A-1 cls별 / A-2 confidence calibration[forward] / A-3 soft-cap)
#   재생성 없이 train_recs(또는 jsonl)만으로 실행. text key = _TEXT_KEY.
# ============================================================
_NUMERIC_F = ("purchase_value", "age")
_CAT_F     = ("sex", "source", "browser", "ip_country", "weekday_purchase", "month_purchase")
def _norm_key(text, n=40): return re.sub(r"[^a-z0-9 ]", "", (text or "").lower())[:n]

# ── A-1. cls별 분리 진단 ──
def per_cls_report(records):
    rows = {}
    for gt, name in [(0, "NORMAL"), (1, "ABNORMAL")]:
        rs = [(json.loads(r["assistant"]).get(_TEXT_KEY) or "").strip().lower()
              for r in records if r["metadata"]["gt_label"] == gt]
        rs = [t for t in rs if t]                                  # null 제외
        c = Counter(rs); ck = Counter(_norm_key(t) for t in rs); n = len(rs)
        num = sum(any(f in t for f in _NUMERIC_F) for t in rs) / max(n, 1)
        cat = sum(any(f in t for f in _CAT_F)     for t in rs) / max(n, 1)
        rows[name] = dict(n=n, uniq=len(c)/max(n,1),
                          top1=c.most_common(1)[0][1]/max(n,1) if c else 0,
                          top1_soft=ck.most_common(1)[0][1]/max(n,1) if ck else 0,
                          num_cite=num, cat_cite=cat)
        print(f"[{name}] n={n}  고유율={rows[name]['uniq']:.1%}  top1={rows[name]['top1']:.1%}  "
              f"soft-top1(40자키)={rows[name]['top1_soft']:.1%}")
        print(f"   numeric 인용율={num:.1%}  categorical 인용율={cat:.1%}")
        print(f"   최빈 3: {[(t[:52], k) for t, k in c.most_common(3)]}")
    print("   판정: NORMAL soft-top1 >15% → B-1(lowest-|z|) / cat_cite <20% → B-2(categorical) 필요")
    return rows

_ = per_cls_report(train_recs)

# ── A-2. 재라벨 confidence calibration (forward 전용) ──
def confidence_calibration(records, n_bins=5):
    confs, agree = [], []
    for r in records:
        nd = float(r["metadata"]["top1_l2_dist_norm"])
        confs.append(json.loads(r["assistant"])["confidence"])
        agree.append(int((1 if nd >= 0.5 else 0) == int(r["metadata"]["gt_label"])))
    confs, agree = np.array(confs), np.array(agree)
    print(f"\n[A-2] confidence-calibration (n={len(confs)}, nd-GT 전체 일치율={agree.mean():.1%})")
    bins = np.quantile(confs, np.linspace(0, 1, n_bins + 1)); bins[-1] += 1; ece = 0.0
    for i in range(n_bins):
        sel = (confs >= bins[i]) & (confs < bins[i+1])
        if sel.sum() == 0: continue
        acc, cbar = agree[sel].mean(), confs[sel].mean() / 100.0
        ece += sel.mean() * abs(acc - cbar)
        print(f"  conf [{bins[i]:5.1f},{bins[i+1]:5.1f}) n={sel.sum():4d}  평균conf={cbar:.2f}  "
              f"nd-GT 일치율={acc:.2f}  gap={acc-cbar:+.2f}")
    n_ndwrong = sum(1 for r in records
                    if (float(r["metadata"]["top1_l2_dist_norm"]) >= 0.5) != bool(r["metadata"]["gt_label"]))
    print(f"  ECE(라벨 함수) = {ece:.3f}   nd 반대편 정답 레코드: {n_ndwrong} ({n_ndwrong/max(len(records),1):.1%})")
    return ece

if TEACHER_TASK == "forward":
    _ = confidence_calibration(train_recs)
else:
    print("\n[A-2] agent 모드: confidence 없음 → calibration 생략 (F4의 wm 형식 준수율로 대체)")

# ── A-3. NORMAL near-dup soft-cap (task 공용) ──
def soft_cap_normal(records, cap_frac=0.02):
    cap = max(3, int(len(records) * cap_frac)); seen = {}; kept, dropped = [], 0
    for r in records:
        if r["metadata"]["gt_label"] == 1: kept.append(r); continue
        txt = json.loads(r["assistant"]).get(_TEXT_KEY)
        if not txt: kept.append(r); continue                       # null wm은 캡 제외
        k = _norm_key(txt); seen[k] = seen.get(k, 0) + 1
        (kept.append(r) if seen[k] <= cap else None)
        if seen[k] > cap: dropped += 1
    print(f"\n[A-3] NORMAL soft-cap(키당 ≤{cap}): {len(records)} → {len(kept)} (제외 {dropped})")
    return kept

train_recs_capped = soft_cap_normal(train_recs)
_save_jsonl(train_recs_capped, OUT_JSONL_TASK.replace(".jsonl", "_capped.jsonl"))


[NORMAL] n=1837  고유율=64.6%  top1=1.7%  soft-top1(40자키)=4.8%
   numeric 인용율=75.2%  categorical 인용율=39.6%
   최빈 3: [('normalized distance 0.31 close to neighbor, purchase', 31), ('normalized distance 0.36 close to neighbor, purchase', 29), ('normalized distance 0.34 close to neighbor, purchase', 28)]
[ABNORMAL] n=1163  고유율=97.1%  top1=0.3%  soft-top1(40자키)=3.8%
   numeric 인용율=78.2%  categorical 인용율=31.0%
   최빈 3: [('normalized distance 0.53 indicates moderate similari', 4), ('high normalized distance 0.69, purchase_value and ag', 3), ('high normalized distance 0.67, purchase_value and ag', 3)]
   판정: NORMAL soft-top1 >15% → B-1(lowest-|z|) / cat_cite <20% → B-2(categorical) 필요

[A-2] confidence-calibration (n=3000, nd-GT 전체 일치율=83.7%)
  conf [ 70.0, 80.0) n=  31  평균conf=0.73  nd-GT 일치율=0.39  gap=-0.35
  conf [ 80.0, 86.0) n=2969  평균conf=0.81  nd-GT 일치율=0.84  gap=+0.03
  ECE(라벨 함수) = 0.036   nd 반대편 정답 레코드: 488 (16.3%)

[A-3] NORMAL soft-cap(키당 ≤60): 3000 → 2936 (제외 64)
  💾 저장: /content/gc

In [22]:
import re
from collections import Counter
_DIRP = re.compile(r"^(cluster_mismatch|neighbor_insufficient)\s*:")
def _dev_of(r):
    md=r["metadata"]; return build_deviation_table(extract_test_features(int(md["test_idx"])),
                                                   CLUSTER_PERSONAS.get(int(md["cluster_id"])))
def _cited(t,dev): t=(t or"").lower(); return {c for c in list(dev["numeric"])+list(dev["categorical"]) if c.lower() in t}
def _retr(u):
    m=re.search(r"agrees:\s*([^|]*)\|\s*differs:\s*([^\n]*)",u or"")
    _n=lambda s:(0 if(not s.strip() or s.strip().startswith("(none)"))else len([x for x in s.split(",")if x.strip()]))
    return "[spread]" in(u or""), (_n(m.group(1))if m else 0), (_n(m.group(2))if m else 0)

def validate_dataset(records, tag="train"):
    P=[(r,json.loads(r["assistant"])) for r in records]
    conf=[(r,a) for r,a in P if a.get("sufficiency")=="confident"]
    amb =[(r,a) for r,a in P if a.get("sufficiency")=="ambiguous"]; n=max(len(amb),1)
    print(f"=== [F6] {tag} n={len(P)} confident={len(conf)} ambiguous={len(amb)} ===")
    print(f"[1] coupling: confident&null {sum(a.get('what_missing')is None for _,a in conf)}/{len(conf)} | "
          f"ambiguous&wm {sum(bool(a.get('what_missing'))for _,a in amb)}/{len(amb)}")
    gt=lambda r:int(r['metadata']['gt_label']); ndv=lambda r:float(r['metadata']['top1_l2_dist_norm'])
    e=sum((a.get('cls')=="ABNORMAL")==bool(gt(r))for r,a in amb)
    b=sum((ndv(r)>=0.5)==bool(gt(r))for r,_ in amb)
    df=sum((a.get('cls')=="ABNORMAL")!=(ndv(r)>=0.5)for r,a in amb)
    print(f"[2] ambiguous cls: evidence-GT {e/n:.1%} vs nd-GT {b/n:.1%} | evidence≠nd {df}/{len(amb)} ← agent 개입/가치")
    dirs=Counter(); cm=cmk=ni=nik=0
    for r,a in amb:
        wm=a.get("what_missing")
        if not wm: continue
        low=wm.lower(); mm=_DIRP.match(wm); dirs[mm.group(1)if mm else"other"]+=1
        if low.startswith("cluster_mismatch"):
            dev=_dev_of(r); cm+=1; cmk+=int(any(c in dev["deviant"]for c in _cited(wm,dev)))
        elif low.startswith("neighbor_insufficient"):
            sp,ag,dfr=_retr(r["user"]); ni+=1; nik+=int(sp or dfr>=ag)
    print(f"[3] direction {dict(dirs)}")
    print(f"    cluster_mismatch grounded(persona 이탈)        {cmk}/{cm}={cmk/max(cm,1):.1%}")
    print(f"    neighbor_insufficient grounded(spread/differs) {nik}/{ni}={nik/max(ni,1):.1%}")

validate_dataset(train_recs, "train")

=== [F6] train n=3000 confident=0 ambiguous=0 ===
[1] coupling: confident&null 0/0 | ambiguous&wm 0/0
[2] ambiguous cls: evidence-GT 0.0% vs nd-GT 0.0% | evidence≠nd 0/0 ← agent 개입/가치
[3] direction {}
    cluster_mismatch grounded(persona 이탈)        0/0=0.0%
    neighbor_insufficient grounded(spread/differs) 0/0=0.0%


In [23]:
# ── [F7] direction GT 정합성 진단 (학습 전 필수) ──
import json
import numpy as np
from collections import Counter

# ★ sufficiency 키에 의존하지 않음 — nd에서 직접 유도 (1-key 전환/stale 대비)
_ndof  = lambda r: float(r["metadata"].get("top1_l2_dist_norm", -1))
_sufof = lambda r: (json.loads(r["assistant"]).get("sufficiency")
                    or (_suf_from_nd(_ndof(r)) if _ndof(r) >= 0 else None))
_wmof  = lambda r: json.loads(r["assistant"]).get("what_missing")

_amb = [r for r in train_recs if _sufof(r) == "ambiguous"]
print(f"\n=== [F7] train={len(train_recs)}  ambiguous={len(_amb)} ===")

if not _amb:
    print("⚠ ambiguous 0건 — 진단:")
    print("   sufficiency 값:", dict(Counter(json.loads(r["assistant"]).get("sufficiency", "<none>")
                                             for r in train_recs[:100])))
    _nds = [_ndof(r) for r in train_recs if _ndof(r) >= 0]
    if _nds:
        print(f"   nd 범위: {min(_nds):.3f} ~ {max(_nds):.3f} | "
              f"밴드[{SUF_LOW},{SUF_HIGH}) 내: {sum(SUF_LOW <= x < SUF_HIGH for x in _nds)}건")
    print("   → 0이면 재생성 전 stale train_recs 이거나 밴드 밖만 채택된 것")
else:
    # [1] ambiguous의 실제 구성
    print("[1] mining_type:", dict(Counter(r["metadata"].get("_mining_type", "?") for r in _amb)))
    print("    'clean' 대다수 → ambiguous(nd 밴드) ≠ 검색 결함 = wm의 전제가 성립 안 함")

    # [2] teacher direction vs GT — ★ mined 서브셋에서만 유효
    #     (ROUTER_CLUSTER/MINING_TYPE 맵이 mined 샘플만 커버하므로)
    _mined = [r for r in _amb if r["metadata"].get("_mining_type", "clean") != "clean"]
    print(f"    mined 서브셋 n={len(_mined)} ← [2]는 여기서만 유효")
    for _dir, _gtk in [("cluster_mismatch", "_misroute"), ("neighbor_insufficient", "_retr_error")]:
        if not _mined:
            print(f"[2] {_dir:22s} skip (mined 0건)"); continue
        yp = [str(_wmof(r)).startswith(_dir) for r in _mined]
        yt = [bool(r["metadata"].get(_gtk, 0)) for r in _mined]
        tp = sum(a and b for a, b in zip(yt, yp))
        fp = sum((not a) and b for a, b in zip(yt, yp))
        fn = sum(a and (not b) for a, b in zip(yt, yp))
        _base = sum(yt) / max(len(yt), 1)
        _P    = tp / max(tp + fp, 1)
        print(f"[2] {_dir:22s} P={_P:.3f} R={tp/max(tp+fn,1):.3f} "
              f"| base={_base:.3f} 예측율={sum(yp)/max(len(yp),1):.3f} "
              f"→ {'✅ 신호 있음' if _P > _base + 0.05 else '❌ base rate 이하 = 정보량 0'}")

    # [3] deviation이 misroute를 예측하나 (가설의 직접 검정)
    _pairs = [(len(_dev_of(r)["deviant"]), int(r["metadata"].get("_misroute", 0))) for r in _amb]
    _a = np.array(_pairs, dtype=float).reshape(-1, 2)          # ★ 빈 배열/1-D 방어
    if _a.shape[0] and _a[:, 1].sum() > 0 and (_a[:, 1] == 0).sum() > 0:
        _m1, _m0 = _a[_a[:, 1] == 1, 0].mean(), _a[_a[:, 1] == 0, 0].mean()
        print(f"[3] deviant 개수 평균: misroute={_m1:.2f} vs 정상배정={_m0:.2f} (Δ={_m1-_m0:+.2f})")
        print("    Δ≈0 → 프롬프트에 misroute 신호 부재 (router cluster 미적용) → 학습 불가")
        print("    Δ>0 → 배정 오류가 deviation으로 드러남 = 가설 성립 → 학습 진행 가능")
    else:
        print(f"[3] skip — misroute={int(_a[:,1].sum()) if _a.shape[0] else 0}, "
              f"정상배정={int((_a[:,1]==0).sum()) if _a.shape[0] else 0} (양쪽 다 있어야 비교 가능)")


=== [F7] train=3000  ambiguous=1942 ===
[1] mining_type: {'type1_misrouted': 199, 'type2_cross_cluster': 231, 'type3_distance_band': 12, 'clean': 1500}
    'clean' 대다수 → ambiguous(nd 밴드) ≠ 검색 결함 = wm의 전제가 성립 안 함
    mined 서브셋 n=442 ← [2]는 여기서만 유효
[2] cluster_mismatch       P=0.000 R=0.000 | base=0.450 예측율=0.000 → ❌ base rate 이하 = 정보량 0
[2] neighbor_insufficient  P=0.000 R=0.000 | base=0.523 예측율=0.000 → ❌ base rate 이하 = 정보량 0
[3] deviant 개수 평균: misroute=6.62 vs 정상배정=6.60 (Δ=+0.02)
    Δ≈0 → 프롬프트에 misroute 신호 부재 (router cluster 미적용) → 학습 불가
    Δ>0 → 배정 오류가 deviation으로 드러남 = 가설 성립 → 학습 진행 가능


## teacher모델 저장

In [ ]:
# ── [최종] Teacher 모델 GCS 저장 (SAVE_TEACHER_TO_GCS=True 일 때만) ──
SAVE_TEACHER_TO_GCS = True   # ★ 저장 스위치

if SAVE_TEACHER_TO_GCS and TEACHER_PROVIDER == "huggingface":
    import os
    _name = HF_TEACHER_MODEL.rstrip("/").split("/")[-1]     # 예: Qwen2.5-14B-Instruct-AWQ
    _dst  = f"/content/gcs_mount/qwen/{_name}"
    os.makedirs(_dst, exist_ok=True)
    hf_model.save_pretrained(_dst, safe_serialization=True)
    hf_tokenizer.save_pretrained(_dst)
    _sz = sum(os.path.getsize(os.path.join(r, f))
              for r, _, fs in os.walk(_dst) for f in fs) / 1024**3
    print(f"✅ Teacher 저장: {_dst}  ({_sz:.1f} GB, {HF_TEACHER_MODEL})")
else:
    print(f"⏭ Teacher 저장 skip (SAVE_TEACHER_TO_GCS={SAVE_TEACHER_TO_GCS}, provider={TEACHER_PROVIDER})")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## NB02 / NB03 연계 안내
**학습 (NB02 재사용)**:
- `TRAINING_JSONL_PATH = OUT_BASE + "/training_data_forward.jsonl"` 로 교체
- `to_chatml`의 `_KEEP_KEYS = ("cls", "confidence", "reasoning")` — 3-key 전체가
  학습 타깃 (forward는 전부-LLM baseline이 전제이므로 WM_FOCUS류 주입 없음)
- BASE_MODEL은 agent evaluator와 동일하게 (0.5B/3B) → GGUF 변환 → GCS 업로드

**평가 (NB03 재사용)**:
- Section A(heuristic): `what_missing` 자리에 `reasoning` — feature 인용율·boilerplate·dedup
- Section B(judge): 동일 rubric으로 reasoning grounded 채점 + judge-GT 상관 검증
- **추가 축 — confidence ECE**: 학습 모델의 confidence vs GT 일치율 calibration
  (forwardRAG의 "평균 회귀" 한계를 재라벨 학습이 얼마나 교정했는지가 핵심 비교점)
- Section C(시스템): GKE에서 `PIPELINE_MODE=forward` + `ANALYZER_TASK=forward`로
  serving 후 동일 eval set 배치 → agent 모드와 버전 비교 표의 baseline row